# Project 1. Seq2Seq Num2Text (8팀)

num, en, ko 6개 방향 번역기. 방향마다 GRU encoder-decoder 모델 1개.

| 담당 | 방향 | 구현 |
| --- | --- | --- |
| 박정연 | - | 베이스라인, 통합 |
| 김태우 | num2en | Bahdanau attention |
| 김한나 | en2num | 양방향 encoder |
| 홍정표 | num2ko | beam 길이 패널티 |
| 김서현 | ko2num | 짧은 숫자 오버샘플링 |
| 김수현 | en2ko | en-ko 데이터 증강 |
| 이재노 | ko2en | 입력 정규화 |

- 공통 베이스라인을 복사해서 방향별로 학습, 각자 브랜치/PR로 제출
- 방향별 학습 로그, 실험 전체는 `work/<방향>.ipynb`
- 이 노트북은 공통 코드 + 팀원 구현 통합, 저장된 .pt로 평가, 마지막 셀에서 로드·테스트

In [1]:
# 설정 (project01/ 에서 실행)
DATA_DIR = 'data'   # num2txt.txt, num2kor.txt
MODEL_DIR = '.'     # <방향>.pt
RETRAIN = False     # True면 6개 방향 다시 학습 후 .pt 덮어씀 (오래 걸림)

BATCH_SIZE = 128
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
LP_ALPHA = 0.6
SEED = 42
MAX_TRAIN = None
EPOCHS = 15
MAX_LEN = 40        # 기본값. 실제 디코딩은 각 .pt에 저장된 max_len 사용

In [2]:
# Colab 사용 시
# !git clone https://github.com/jungyy808/2026-nlp.git
# %cd 2026-nlp/project01

In [3]:
import os, random, time, zlib
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. 데이터

In [4]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    # en: 단어 단위 / num, ko: 문자 단위 (공백 포함)
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    # CRLF 제거
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    # (src, tgt, key) 반환, key는 split 기준
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:  # en-ko: 공통 key만 사용
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    # key 해시 기준 8:1:1 분할
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):  # sos/eos 제외
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    # extra_train: 증강 등으로 추가할 (src, tgt, key) 리스트. train에만 들어가고 valid/test는 원본만 사용
    # [김수현] extra_train의 토큰도 vocab에 포함 (extra_train=None이면 베이스라인과 동일)
    # en-ko 공통 숫자는 1~8자리뿐이라 원본만으로는 'billion', '억'이 vocab에 없음 -> 증강 데이터가 <unk>가 됨
    vocab_pairs = pairs + list(extra_train or [])
    src_vocab = Vocab(src_lang, [s for s, _, _ in vocab_pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in vocab_pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]  # valid/test 숫자와 겹치는 건 제외
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)

In [5]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음'
en_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['en'])))
ko_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['ko'])))
print(f'num2txt {len(en_all):,}줄, num2kor {len(ko_all):,}줄, 공통 숫자 {len(set(en_all) & set(ko_all)):,}개')
for k in ['4507', '21546']:
    print(f'  {k} -> {en_all.get(k)} / {ko_all.get(k)}')

작업 위치: /home/claude/merge2/project01


num2txt 100,000줄, num2kor 100,000줄, 공통 숫자 12,738개
  4507 -> four thousand five hundred seven / 사천오백칠
  21546 -> None / 이만 천오백사십육


In [6]:
# [김서현] 짧은 숫자(1~3자리) 오버샘플링 (ko2num)
# 1~3자리 데이터가 적어서 train에서 자릿수별 target개까지 반복 (valid/test는 그대로)


def oversample_short(train_pairs, max_digits=3, target=3000):
    # 추가로 넣을 쌍(복제본)만 반환한다. 원본은 이미 train에 있으므로 포함하지 않는다.
    groups = defaultdict(list)
    for p in train_pairs:
        if len(p[2]) <= max_digits:
            groups[len(p[2])].append(p)
    extra = []
    for L, group in sorted(groups.items()):
        need = target - len(group)
        if need <= 0:
            continue
        reps, rem = divmod(need, len(group))
        # 전체를 reps번 반복하고, 남는 개수는 무작위로 골라 채운다
        extra += group * reps + random.Random(SEED + L).sample(group, rem)
    return extra

## 2. 모델

- `bidirectional` (김한나), `attention='bahdanau'` (김태우)를 공통 클래스에 CONFIG 옵션으로 합침
- 기본값 `attention='luong'`, `bidirectional=False`

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0, bidirectional=bidirectional)
        # [김한나] 두 방향의 2H 표현을 기존 decoder/attention의 H 차원으로 연결
        self.bidirectional = bidirectional
        self.n_layers, self.hid_dim = n_layers, hid_dim
        if bidirectional:
            self.output_bridge = nn.Linear(2 * hid_dim, hid_dim)
            self.hidden_bridge = nn.Linear(2 * hid_dim, hid_dim)
            if rnn == 'lstm':
                self.cell_bridge = nn.Linear(2 * hid_dim, hid_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        if self.bidirectional:
            out = self.output_bridge(out)
            def bridge(h, projection):
                # [layers*2,B,H] -> [layers,B,2H] -> [layers,B,H]
                h = h.reshape(self.n_layers, 2, h.size(1), self.hid_dim)
                return torch.tanh(projection(torch.cat((h[:, 0], h[:, 1]), dim=-1)))
            if isinstance(hidden, tuple):
                hidden = (bridge(hidden[0], self.hidden_bridge),
                          bridge(hidden[1], self.cell_bridge))
            else:
                hidden = bridge(hidden, self.hidden_bridge)
        return out, hidden


class Decoder(nn.Module):
    # [김태우] attention='bahdanau' 분기 (num2en)
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention):
        super().__init__()
        assert attention in ('luong', 'bahdanau', None), f'지원하지 않는 attention: {attention}'
        assert rnn in ('gru', 'lstm'), f'지원하지 않는 rnn: {rnn}'
        self.attention = attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        rnn_in = emb_dim + hid_dim if attention == 'bahdanau' else emb_dim
        self.rnn = RNN(rnn_in, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)  # Luong (general)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        elif attention == 'bahdanau':
            self.W_h = nn.Linear(hid_dim, hid_dim, bias=False)
            self.W_s = nn.Linear(hid_dim, hid_dim, bias=False)
            self.v_a = nn.Linear(hid_dim, 1, bias=False)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)
        if self.attention == 'bahdanau':
            query = hidden[0][-1] if isinstance(hidden, tuple) else hidden[-1]
            scores = self.v_a(torch.tanh(self.W_h(enc_out) + self.W_s(query).unsqueeze(1))).squeeze(-1)
            scores = scores.masked_fill(~src_mask, -1e9)
            ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
            h, hidden = self.rnn(torch.cat([x, ctx.unsqueeze(1)], dim=2), hidden)
            h = torch.tanh(self.combine(torch.cat([h.squeeze(1), ctx], dim=1)))
        else:
            h, hidden = self.rnn(x, hidden)
            h = h.squeeze(1)
            if self.attention == 'luong':
                scores = torch.bmm(enc_out, self.attn(h).unsqueeze(2)).squeeze(2)
                scores = scores.masked_fill(~src_mask, -1e9)
                ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
                h = torch.tanh(self.combine(torch.cat([h, ctx], dim=1)))
        return self.out(self.drop(h)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong', bidirectional=False):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, bidirectional)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)  # (hidden, enc_out, mask)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        # tgt[:, 1:] 예측
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    # beam search용 배치 재정렬
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)

## 3. 학습, 디코딩

- greedy, beam (`beam_decode_lp`, 홍정표 길이 패널티 `((5+L)/6)^alpha`)

In [8]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    # batch greedy
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)  # 로드한 모델은 저장된 max_len 사용
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    # exact match 기준
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR):
    # best valid acc 모델 복원
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history

In [9]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')


# [홍정표] 길이 패널티 + 안전한 조기 종료 beam search (num2ko)
@torch.no_grad()
def beam_decode_lp(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, alpha=LP_ALPHA, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams, finished = [([], 0.0)], []

    def length_penalty(length):
        return ((5.0 + length) / 6.0) ** alpha

    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp_val, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp_val, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / length_penalty(len(seq))))
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams:
            break
        if finished:  # 남은 beam이 더 좋아질 수 없을 때만 종료
            if beams[0][1] / length_penalty(max_len) <= max(f[1] for f in finished):
                break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / length_penalty(len(seq))) for seq, score in beams]
    return tgt_vocab.decode(max(finished, key=lambda f: f[1])[0])

In [10]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    # 저장 형식 고정 (통합 시 로드)
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt

## 4. en-ko 데이터 증강 (김수현)

- 한쪽 파일에만 있는 숫자를 num2en, num2ko 모델로 채워서 en-ko 쌍 생성
- 생성 데이터는 train에만 사용, test는 원본

In [11]:
# [김수현] en-ko 증강: 한쪽에만 있는 train 숫자의 빈 언어를 num2en/num2ko로 생성

@torch.no_grad()
def translate_batch(model, texts, sv, tv, batch_size=1024):
    # 여러 입력을 배치 greedy로 번역 (증강 데이터 생성용)
    outs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        src, src_len, _ = collate([(sv.encode(t), [SOS]) for t in chunk])
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        outs += [tv.decode(p) for p in pred.tolist()]
    return outs


def find_pt(name, save_dir=MODEL_DIR):
    # project01/ 에 없으면 Colab 업로드 위치(/content, 현재 폴더)도 확인
    for d in (save_dir, '.', '/content'):
        p = os.path.join(d, f'{name}.pt')
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name}.pt 없음: project01/에 merge됐는지 확인하거나 Colab 왼쪽 폴더에 업로드')


def build_augmented_pairs(data_dir=DATA_DIR, save_dir=MODEL_DIR):
    en = dict(load_pairs(os.path.join(data_dir, FILES['en'])))
    ko = dict(load_pairs(os.path.join(data_dir, FILES['ko'])))
    only_en = [k for k in en if k not in ko and split_of(k) == 'train']
    only_ko = [k for k in ko if k not in en and split_of(k) == 'train']

    gen_ko_model = load_model(find_pt('num2ko', save_dir))  # (model, sv, tv, ckpt)
    gen_en_model = load_model(find_pt('num2en', save_dir))
    assert gen_ko_model[1].lang == 'num' and gen_ko_model[2].lang == 'ko', 'num2ko.pt 언어 방향이 올바르지 않습니다.'
    assert gen_en_model[1].lang == 'num' and gen_en_model[2].lang == 'en', 'num2en.pt 언어 방향이 올바르지 않습니다.'
    t0 = time.time()
    gen_ko = translate_batch(gen_ko_model[0], only_en, *gen_ko_model[1:3])
    gen_en = translate_batch(gen_en_model[0], only_ko, *gen_en_model[1:3])
    print(f'생성 시간: {time.time() - t0:.0f}s')

    extra = ([(en[k], g, k) for k, g in zip(only_en, gen_ko)] +   # 영어 원문 + 한글 생성
             [(g, ko[k], k) for k, g in zip(only_ko, gen_en)])    # 영어 생성 + 한글 원문
    print(f'영어만 있는 train 숫자 {len(only_en):,}개 -> num2ko로 한글 생성')
    print(f'한글만 있는 train 숫자 {len(only_ko):,}개 -> num2en으로 영어 생성')
    print(f'증강 쌍 합계 {len(extra):,}개')
    return extra, gen_en_model, gen_ko_model

## 5. 입력 정규화 (이재노)

- `thousands` → `thousand`, `일만` → `만` 등 데이터 표기로 변환

In [12]:
import re

# [이재노] 입력 정규화 (ko2en)

def normalize_ko(text):
    s = re.sub(r'[\s,]', '', text)          # 공백, 쉼표 제거
    s = re.sub(r'일([천백십])', r'\1', s)     # 일천 -> 천
    s = re.sub(r'(^|억)일만', r'\1만', s)     # 일만 -> 만 (십일만은 그대로)
    s = re.sub(r'^억', '일억', s)             # 억 -> 일억
    s = re.sub(r'([억만])(?=.)', r'\1 ', s)   # 억, 만 뒤 띄어쓰기
    return s

def normalize_en(text):
    s = re.sub(r'[-,]', ' ', text.lower())
    s = re.sub(r'\band\b', ' ', s)
    s = re.sub(r'(thousand|hundred|million|billion)s\b', r'\1', s)
    return ' '.join(s.split())

def normalize_num(text):
    return re.sub(r'[\s,_]', '', text)

def normalize(text, lang):
    if lang == 'ko':
        return normalize_ko(text)
    if lang == 'en':
        return normalize_en(text)
    return normalize_num(text)


for text, lang in [('일만이천삼백사십오', 'ko'), ('일억일만', 'ko'), ('십일만 일천', 'ko'),
                   ('twelve thousands three hundreds forty five', 'en'),
                   ('One Hundred and Twenty-One', 'en'), ('12,345', 'num')]:
    print(f'{lang:3s} {text!r:45s} -> {normalize(text, lang)!r}')

ko  '일만이천삼백사십오'                                   -> '만 이천삼백사십오'
ko  '일억일만'                                        -> '일억 만'
ko  '십일만 일천'                                      -> '십일만 천'
en  'twelve thousands three hundreds forty five'  -> 'twelve thousand three hundred forty five'
en  'One Hundred and Twenty-One'                  -> 'one hundred twenty one'
num '12,345'                                      -> '12345'


## 6. 방향별 학습 설정

- `RETRAIN = False`면 업로드된 .pt 사용 (학습 로그는 `work/`)

In [13]:
BASE = dict(emb_dim=128, hid_dim=256, n_layers=2, dropout=0.2, rnn='gru', attention='luong')
SMALL = dict(BASE, hid_dim=128, dropout=0.3)  # en-ko는 데이터가 적어서 작게

SETTINGS = {
    # 방향: (CONFIG, epochs, 추가 데이터, 담당)
    'num2en': (dict(BASE, attention='bahdanau'), 15, 'oversample2', '김태우'),
    'en2num': (dict(BASE, bidirectional=True), 15, None, '김한나'),
    'num2ko': (BASE, 15, None, '홍정표'),
    'ko2num': (BASE, 15, 'oversample', '김서현'),
    'en2ko': (SMALL, 10, 'augment', '김수현'),
    'ko2en': (SMALL, 10, 'augment_rev', '이재노'),
}
DIRECTIONS = list(SETTINGS)


def split_direction(d):
    src, tgt = d.split('2')
    return src, tgt


def train_direction(d):
    global MAX_LEN, tgt_vocab
    src, tgt = split_direction(d)
    config, epochs, extra_kind, who = SETTINGS[d]
    set_seed(SEED)
    pairs = get_pairs(src, tgt)
    extra = None
    train_pairs = [p for p in pairs if split_of(p[2]) == 'train']
    if extra_kind == 'oversample':        # 김서현: 1~3자리
        extra = oversample_short(train_pairs)
    elif extra_kind == 'oversample2':     # 김태우: 1~2자리
        extra = oversample_short(train_pairs, max_digits=2)
    elif extra_kind in ('augment', 'augment_rev'):  # 김수현 증강 (이재노는 방향만 뒤집어서 사용)
        extra, _, _ = build_augmented_pairs()
        if extra_kind == 'augment_rev':
            extra = [(t, s, k) for s, t, k in extra]
        extra = extra + [p for p in train_pairs if len(p[2]) <= 3] * 20
    train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
        pairs, src, tgt, BATCH_SIZE, MAX_TRAIN, extra_train=extra)
    MAX_LEN = max(len(tokenize(t, tgt)) for _, t, _ in pairs + list(extra or [])) + 10
    model = Seq2Seq(len(src_vocab), len(tgt_vocab), **config).to(DEVICE)
    history = train(model, train_loader, valid_loader, epochs, LR)
    rows = predict_all(model, test_loader, tgt_vocab)
    save_model(os.path.join(MODEL_DIR, f'{d}.pt'), model, config, src_vocab, tgt_vocab,
               metrics={'test_exact_match': exact_match(rows), 'history': history})


if RETRAIN:
    for d in DIRECTIONS:
        print(f'===== {d} ({SETTINGS[d][3]}) =====')
        train_direction(d)
else:
    print('RETRAIN = False: 업로드된 .pt 사용')
    for d in DIRECTIONS:
        p = os.path.join(MODEL_DIR, f'{d}.pt')
        print(f'  {d:7s} {SETTINGS[d][3]}  {"있음" if os.path.exists(p) else "없음"}')

RETRAIN = False: 업로드된 .pt 사용
  num2en  김태우  있음
  en2num  김한나  있음
  num2ko  홍정표  있음
  ko2num  김서현  있음
  en2ko   김수현  있음
  ko2en   이재노  있음


## 7. test 평가

- test: 숫자 해시 기준 10%, greedy exact match

In [14]:
MODELS = {}
for d in DIRECTIONS:
    p = os.path.join(MODEL_DIR, f'{d}.pt')
    if os.path.exists(p):
        MODELS[d] = load_model(p)  # (model, src_vocab, tgt_vocab, ckpt)
print('로드:', ', '.join(f"{d}({sum(x.numel() for x in MODELS[d][0].parameters()):,})" for d in MODELS))
missing = [d for d in DIRECTIONS if d not in MODELS]
if missing:
    print('없음:', missing)


def test_pairs_of(d):
    src, tgt = split_direction(d)
    return [p for p in get_pairs(src, tgt) if split_of(p[2]) == 'test']


@torch.no_grad()
def evaluate_saved(d, batch_size=512):
    model, sv, tv, _ = MODELS[d]
    pairs = test_pairs_of(d)
    rows = []
    for i in range(0, len(pairs), batch_size):
        chunk = pairs[i:i + batch_size]
        src, src_len, _ = collate([(sv.encode(s), [SOS]) for s, _, _ in chunk])
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        rows += [(s, t, tv.decode(p), k) for (s, t, k), p in zip(chunk, pred.tolist())]
    return rows


RESULTS = {}

로드: num2en(1,857,316), en2num(2,936,334), num2ko(1,588,756), ko2num(1,587,214), en2ko(455,316), ko2en(457,380)


### num2en (김태우, Bahdanau attention)

| 모델 | valid | test |
| --- | --- | --- |
| Luong | 99.65% | 99.85% |
| Bahdanau | 100.00% | 99.98% |
| attention 없음 | 99.88% | 99.85% |

- 한 자리 오답(`0`→`ten`, `6`→`sixty`)이 있어서 1~2자리 오버샘플링 후 재학습 (test 99.97%)
- 제출: Bahdanau + 오버샘플링

In [15]:
d = 'num2en'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

num2en test exact match (greedy): 99.97%  (n=10,005)
config: {'emb_dim': 128, 'hid_dim': 256, 'n_layers': 2, 'dropout': 0.2, 'rnn': 'gru', 'attention': 'bahdanau'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 | 100.00%
     8 |   1472 | 100.00%
     9 |   1553 |  99.87%
    10 |   1448 |  99.93%

--- 맞은 예시 ---
입력: 897890
정답: eight hundred ninety seven thousand eight hundred ninety
예측: eight hundred ninety seven thousand eight hundred ninety

입력: 93532
정답: ninety three thousand five hundred thirty two
예측: ninety three thousand five hundred thirty two

입력: 3894895050
정답: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
예측: three billion eight hundred ninety four million eight hundred ninety five thousand fifty

--- 틀린 예시 ---
입력: 410100575
정답: four hundred ten million one hundred thousand five hundred sevent

### en2num (김한나, 양방향 encoder)

| 모델 | 파라미터 | valid | test |
| --- | --- | --- | --- |
| 단방향 | 1,589,262 | 99.96% | 99.96% |
| 양방향 | 2,936,334 | 99.98% | 99.96% |

- valid 기준으로 양방향 제출

In [16]:
d = 'en2num'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

en2num test exact match (greedy): 99.96%  (n=10,005)
config: {'emb_dim': 128, 'hid_dim': 256, 'n_layers': 2, 'dropout': 0.2, 'rnn': 'gru', 'bidirectional': True, 'attention': 'luong'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 | 100.00%
     8 |   1472 | 100.00%
     9 |   1553 |  99.87%
    10 |   1448 |  99.86%

--- 맞은 예시 ---
입력: eight hundred ninety seven thousand eight hundred ninety
정답: 897890
예측: 897890

입력: ninety three thousand five hundred thirty two
정답: 93532
예측: 93532

입력: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
정답: 3894895050
예측: 3894895050

--- 틀린 예시 ---
입력: three hundred forty eight million sixty
정답: 348000060
예측: 348000663

입력: six billion seven hundred ninety one million six thousand three hundred fifty four
정답: 6791006354
예측: 6791106354

입력: nine hundred ninety one million five

### num2ko (홍정표, beam 길이 패널티)

| 디코딩 (test 500개) | 정확도 |
| --- | --- |
| greedy | 100.00% |
| 기존 beam 5 | 95.80% |
| 길이 패널티 beam 5 | 100.00% |

- 기존 beam은 짧은 후보가 먼저 끝나면 탐색이 종료돼서 뒷부분이 잘림

In [17]:
d = 'num2ko'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

num2ko test exact match (greedy): 99.89%  (n=9,961)
config: {'emb_dim': 128, 'hid_dim': 256, 'n_layers': 2, 'dropout': 0.2, 'rnn': 'gru', 'attention': 'luong'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1491 | 100.00%
     6 |   1497 | 100.00%
     7 |   1508 |  99.54%
     8 |   1493 |  99.87%
     9 |   1474 |  99.93%
    10 |   1502 |  99.93%

--- 맞은 예시 ---
입력: 913269433
정답: 구억 천삼백이십육만 구천사백삼십삼
예측: 구억 천삼백이십육만 구천사백삼십삼

입력: 62198842
정답: 육천이백십구만 팔천팔백사십이
예측: 육천이백십구만 팔천팔백사십이

입력: 144074
정답: 십사만 사천칠십사
예측: 십사만 사천칠십사

--- 틀린 예시 ---
입력: 6897420000
정답: 육십팔억 구천칠백사십이만
예측: 육십팔억 구천칠백사십이만 

입력: 6000830
정답: 육백만 팔백삼십
예측: 육백만 팔천삼십

입력: 14990013
정답: 천사백구십구만 십삼
예측: 천사백구십구만 백삼



### ko2num (김서현, 짧은 숫자 오버샘플링)

| 모델 | 1~3자리 | 전체 test |
| --- | --- | --- |
| 원본 | 100.00% | 99.99% |
| 오버샘플링 | 100.00% | 100.00% |

- 1~3자리를 자릿수별 3,000개까지 반복 (8,194쌍 추가)

In [18]:
d = 'ko2num'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

ko2num test exact match (greedy): 100.00%  (n=9,961)
config: {'emb_dim': 128, 'hid_dim': 256, 'n_layers': 2, 'dropout': 0.2, 'rnn': 'gru', 'attention': 'luong'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1491 | 100.00%
     6 |   1497 | 100.00%
     7 |   1508 | 100.00%
     8 |   1493 | 100.00%
     9 |   1474 | 100.00%
    10 |   1502 | 100.00%

--- 맞은 예시 ---
입력: 구억 천삼백이십육만 구천사백삼십삼
정답: 913269433
예측: 913269433

입력: 육천이백십구만 팔천팔백사십이
정답: 62198842
예측: 62198842

입력: 십사만 사천칠십사
정답: 144074
예측: 144074

--- 틀린 예시 ---


### en2ko (김수현, en-ko 데이터 증강)

| 데이터 | test | 6자리 이상 |
| --- | --- | --- |
| 원본 (10,162쌍) | 99.77% | 46.25% |
| 증강 (166,307쌍) | 100.00% | 99.67% |

- 제출: 증강

In [19]:
d = 'en2ko'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

en2ko test exact match (greedy): 100.00%  (n=1,302)
config: {'emb_dim': 128, 'hid_dim': 128, 'n_layers': 2, 'dropout': 0.3, 'rnn': 'gru', 'attention': 'luong'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |    277 | 100.00%
     6 |     24 | 100.00%
     7 |      4 | 100.00%
     8 |      1 | 100.00%

--- 맞은 예시 ---
입력: one thousand four
정답: 천사
예측: 천사

입력: one thousand ten
정답: 천십
예측: 천십

입력: one thousand eighteen
정답: 천십팔
예측: 천십팔

--- 틀린 예시 ---


### ko2en (이재노, 입력 정규화)

- `일만이천삼백사십오` → `만 이천삼백사십오`, `twelve thousands` → `twelve thousand`
- 6자리 이상 정확도가 낮아서 김수현 증강 데이터를 뒤집어 재학습

| 자릿수 | 원본 | 증강 |
| --- | --- | --- |
| 7 | 9.0% | 99.5% |
| 10 | 0.0% | 99.0% |

- test 99.54% → 99.92%, 제출: 증강

In [20]:
d = 'ko2en'
if d in MODELS:
    rows = evaluate_saved(d)
    RESULTS[d] = rows
    cfg = MODELS[d][3]['config']
    print(f"{d} test exact match (greedy): {exact_match(rows)*100:.2f}%  (n={len(rows):,})")
    print('config:', cfg, '\n')
    acc_by_length(rows)
    print()
    show_examples(rows, n=3)
else:
    print(f'{d}.pt 없음')

ko2en test exact match (greedy): 99.92%  (n=1,302)
config: {'emb_dim': 128, 'hid_dim': 128, 'n_layers': 2, 'dropout': 0.3, 'rnn': 'gru', 'attention': 'luong'} 

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 |  99.89%
     5 |    277 | 100.00%
     6 |     24 | 100.00%
     7 |      4 | 100.00%
     8 |      1 | 100.00%

--- 맞은 예시 ---
입력: 천사
정답: one thousand four
예측: one thousand four

입력: 천십
정답: one thousand ten
예측: one thousand ten

입력: 천십팔
정답: one thousand eighteen
예측: one thousand eighteen

--- 틀린 예시 ---
입력: 천구백팔십팔
정답: one thousand nine hundred eighty eight
예측: one hundred thousand nine hundred eighty eight



## 8. 6개 방향 결과 비교

In [21]:
print(f"{'방향':7s} | {'담당':4s} | {'구조':28s} | {'test 개수':>8s} | {'greedy':>7s} | {'beam5+LP (200개)':>15s}")
print('-' * 90)
for d in DIRECTIONS:
    if d not in RESULTS:
        print(f'{d:7s} | {SETTINGS[d][3]} | (없음)')
        continue
    model, sv, tv, ck = MODELS[d]
    cfg = ck['config']
    arch = f"h{cfg['hid_dim']} {cfg['attention']}" + (' bi' if cfg.get('bidirectional') else '')
    sample = test_pairs_of(d)[:200]
    lp = sum(beam_decode_lp(model, s, sv, tv) == t for s, t, _ in sample) / len(sample)
    print(f"{d:7s} | {SETTINGS[d][3]} | {arch:28s} | {len(RESULTS[d]):8,d} | {exact_match(RESULTS[d])*100:6.2f}% | {lp*100:14.1f}%")

방향      | 담당   | 구조                           |  test 개수 |  greedy | beam5+LP (200개)
------------------------------------------------------------------------------------------


num2en  | 김태우 | h256 bahdanau                |   10,005 |  99.97% |          100.0%


en2num  | 김한나 | h256 luong bi                |   10,005 |  99.96% |          100.0%


num2ko  | 홍정표 | h256 luong                   |    9,961 |  99.89% |          100.0%


ko2num  | 김서현 | h256 luong                   |    9,961 | 100.00% |          100.0%


en2ko   | 김수현 | h128 luong                   |    1,302 | 100.00% |          100.0%


ko2en   | 이재노 | h128 luong                   |    1,302 |  99.92% |           99.5%


## 9. 한계: 0이 많은 숫자

- test 밖 숫자 22개로 추가 테스트, `1000000`, `1000000000` 같은 숫자는 대부분 틀림
- 데이터에 끝자리 0이 5개 이상인 숫자가 없음 (`one million`, `십억`으로 끝나는 문장을 학습한 적 없음)
- 구조, 증강과 관계없이 모든 방향에서 같은 패턴이라 데이터 분포 한계로 봄

In [22]:
for name, table in [('num2txt', en_all), ('num2kor', ko_all)]:
    tz = defaultdict(int)
    for k in table:
        if len(k) > 1:
            tz[len(k) - len(k.rstrip('0'))] += 1
    print(f'{name} 끝자리 0 개수별 숫자 수:', dict(sorted(tz.items())))
print()

HARD = [  # (num, en, ko) 데이터 표기 기준 정답
    ('0', 'zero', '영'), ('6', 'six', '육'), ('7', 'seven', '칠'), ('10', 'ten', '십'), ('11', 'eleven', '십일'),
    ('99', 'ninety nine', '구십구'), ('101', 'one hundred one', '백일'), ('1001', 'one thousand one', '천일'),
    ('10000', 'ten thousand', '만'), ('10001', 'ten thousand one', '만 일'),
    ('100000', 'one hundred thousand', '십만'), ('700000', 'seven hundred thousand', '칠십만'),
    ('1000000', 'one million', '백만'), ('1000001', 'one million one', '백만 일'),
    ('10000000', 'ten million', '천만'), ('30000000', 'thirty million', '삼천만'),
    ('100000000', 'one hundred million', '일억'), ('1000000000', 'one billion', '십억'),
    ('1000000007', 'one billion seven', '십억 칠'), ('2000000000', 'two billion', '이십억'),
    ('1234567890', 'one billion two hundred thirty four million five hundred sixty seven thousand eight hundred ninety',
     '십이억 삼천사백오십육만 칠천팔백구십'),
    ('9999999999', 'nine billion nine hundred ninety nine million nine hundred ninety nine thousand nine hundred ninety nine',
     '구십구억 구천구백구십구만 구천구백구십구'),
]
COL = {'num': 0, 'en': 1, 'ko': 2}
avail = [d for d in DIRECTIONS if d in MODELS]
marks = {}
for d in avail:
    src, tgt = split_direction(d)
    model, sv, tv, _ = MODELS[d]
    marks[d] = [greedy_decode(model, normalize(r[COL[src]], src), sv, tv).strip() == r[COL[tgt]] for r in HARD]
print(f"{'숫자':12s}" + ''.join(f'{d:>8s}' for d in avail))
for i, r in enumerate(HARD):
    print(f'{r[0]:12s}' + ''.join(f"{'O' if marks[d][i] else 'X':>8s}" for d in avail))
print(f"{'맞힌 개수':10s}" + ''.join(f'{sum(marks[d]):>5d}/{len(HARD)}' for d in avail))

num2txt 끝자리 0 개수별 숫자 수: {0: 89956, 1: 9023, 2: 909, 3: 97, 4: 5}
num2kor 끝자리 0 개수별 숫자 수: {0: 90079, 1: 8958, 2: 857, 3: 87, 4: 9}



숫자            num2en  en2num  num2ko  ko2num   en2ko   ko2en
0                  O       O       O       O       O       O
6                  O       O       O       O       O       O
7                  O       O       O       O       O       O
10                 O       O       O       O       O       O
11                 O       O       O       O       O       O
99                 O       O       O       O       O       O
101                O       O       O       O       O       O
1001               O       O       O       O       O       O
10000              O       O       O       X       X       X
10001              O       O       O       O       O       O
100000             O       O       O       X       X       X
700000             O       O       O       X       X       X
1000000            X       X       O       X       X       X
1000001            X       X       O       O       X       X
10000000           X       X       X       X       X       X
30000000           X    

## 10. 모델 로드, 테스트, 사용법

```python
num2en(12345)
en2ko('twelve thousands three hundreds forty five')
ko2num('일만이천삼백사십오', decode='beam')
```
- 함수: `num2en, en2num, num2ko, ko2num, en2ko, ko2en`
- 입력은 문자열 (int를 넣어도 문자열로 변환해서 처리), 정규화 후 번역
- `decode='greedy'` (기본) / `'beam'`

In [23]:
def make_translator(d):
    src, tgt = split_direction(d)
    model, sv, tv, _ = load_model(os.path.join(MODEL_DIR, f'{d}.pt'))

    def translate(text, decode='greedy', beam=BEAM_SIZE):
        text = normalize(str(text), src)  # 문자열로만 처리
        if decode == 'beam':
            return beam_decode_lp(model, text, sv, tv, beam).strip()
        return greedy_decode(model, text, sv, tv).strip()

    translate.__name__ = d
    return translate


for d in DIRECTIONS:
    if os.path.exists(os.path.join(MODEL_DIR, f'{d}.pt')):
        globals()[d] = make_translator(d)
    else:
        print(f'{d}.pt 없음')

tests = [
    ('num2en', 12345), ('num2en', '9876543210'),
    ('en2num', 'twelve thousands three hundreds forty five'),
    ('num2ko', '12345'), ('num2ko', '9876543210'),
    ('ko2num', '일만이천삼백사십오'), ('ko2num', '구십팔억 칠천육백오십사만 삼천이백십'),
    ('en2ko', 'twelve thousands three hundreds forty five'),
    ('ko2en', '일만이천삼백사십오'),
]
for d, x in tests:
    if d in globals():
        f = globals()[d]
        print(f"{d}({x!r})\n  greedy: {f(x)}\n  beam  : {f(x, decode='beam')}")

num2en(12345)
  greedy: twelve thousand three hundred forty five
  beam  : twelve thousand three hundred forty five
num2en('9876543210')
  greedy: nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten
  beam  : nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten
en2num('twelve thousands three hundreds forty five')
  greedy: 12345
  beam  : 12345
num2ko('12345')
  greedy: 만 이천삼백사십오
  beam  : 만 이천삼백사십오
num2ko('9876543210')
  greedy: 구십팔억 칠천육백오십사만 삼천이백십
  beam  : 구십팔억 칠천육백오십사만 삼천이백십
ko2num('일만이천삼백사십오')
  greedy: 12345
  beam  : 12345
ko2num('구십팔억 칠천육백오십사만 삼천이백십')
  greedy: 9876543210
  beam  : 9876543210
en2ko('twelve thousands three hundreds forty five')
  greedy: 만 이천삼백사십오
  beam  : 만 이천삼백사십오
ko2en('일만이천삼백사십오')
  greedy: twelve thousand three hundred forty five
  beam  : twelve thousand three hundred forty five
